In [15]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateGMRES(krylov_size=ks, tol=1e-4, max_iteration=200,n_refinement=2)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = jax.jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")


def run_benchmark(label, single_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve_fn = single_fn
        else:
            solve_fn = lambda p, d, _ks=ks: single_fn(p, d, _ks)

        for mode in ["vmap", "lax.map"]:
            jax.clear_caches()
            gc.collect()

            if mode == "vmap":
                batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
            else:
                batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

            # Forward
            fn_fwd = jax.jit(batched)
            _ = fn_fwd(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                vals = fn_fwd(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_fwd = min(ts) * 1000
            fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

            # Jacobian
            fn_jac = jax.jit(jax.jacfwd(batched))
            _ = fn_jac(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                jac = fn_jac(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_jac = min(ts) * 1000
            jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
            has_nan = bool(jnp.any(jnp.isnan(jac)))

            ks_str = f"{ks}" if ks is not None else "  —"
            nan_tag = " NaN!" if has_nan else ""
            wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
            print(
                f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                f"{jac_err:10.2e}{nan_tag}"
            )

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.



In [11]:
# ── Run benchmarks ───────────────────────────────────────────────
run_benchmark("DENSE SOLVER", dense_single, [None])


DENSE SOLVER
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------
     —      vmap      3004      4255        7260    0.00e+00    0.00e+00
     —   lax.map      2637      2918        5555    0.00e+00    0.00e+00


In [16]:
run_benchmark(
    "GMRES",
    gmres_single,
    [96],
)


GMRES
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------
    96      vmap      1293      4784        6077    8.11e-06    3.29e-06
    96   lax.map      1002      4410        5412    8.40e-06    3.38e-06


In [17]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateArnoldi(krylov_size=ks, tol=1e-6, max_cycles=100,n_refinement=2)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = jax.jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")


def run_benchmark(label, single_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve_fn = single_fn
        else:
            solve_fn = lambda p, d, _ks=ks: single_fn(p, d, _ks)

        for mode in ["vmap", "lax.map"]:
            jax.clear_caches()
            gc.collect()

            if mode == "vmap":
                batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
            else:
                batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

            # Forward
            fn_fwd = jax.jit(batched)
            _ = fn_fwd(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                vals = fn_fwd(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_fwd = min(ts) * 1000
            fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

            # Jacobian
            fn_jac = jax.jit(jax.jacfwd(batched))
            _ = fn_jac(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                jac = fn_jac(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_jac = min(ts) * 1000
            jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
            has_nan = bool(jnp.any(jnp.isnan(jac)))

            ks_str = f"{ks}" if ks is not None else "  —"
            nan_tag = " NaN!" if has_nan else ""
            wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
            print(
                f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                f"{jac_err:10.2e}{nan_tag}"
            )

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.



In [18]:
run_benchmark(
    "Arnoldi",
    gmres_single,
    [96],
)


Arnoldi
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------
    96      vmap     20887     25094       45981    3.16e-07    1.49e-06
    96   lax.map      4986      8402       13388    2.96e-07    1.48e-06


In [ ]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 1
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateGMRES(krylov_size=ks, tol=1e-4, max_iteration=200,n_refinement=2)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real, jnp.max(jnp.abs(dq.to_jax(dq.lindbladian(H_q, Ls_q, result.rho))))


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = jax.jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")

def run_benchmark(label, single_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}  {'res_max':>10}"
    )
    print(f"  {'-' * 95}")

    for ks in krylov_sizes:
        if ks is None:
            solve_fn = single_fn
        else:
            solve_fn = lambda p, d, _ks=ks: single_fn(p, d, _ks)

        for mode in ["vmap", "lax.map"]:
            jax.clear_caches()
            gc.collect()

            if mode == "vmap":
                batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
            else:
                batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

            # observable seulement
            batched_obs = lambda p: batched(p)[0]

            # Forward
            fn_fwd = jax.jit(batched)
            vals, res = fn_fwd(params_true)
            vals.block_until_ready()
            res.block_until_ready()

            ts = []
            for _ in range(3):
                t0 = time.time()
                vals, res = fn_fwd(params_true)
                vals.block_until_ready()
                res.block_until_ready()
                ts.append(time.time() - t0)

            t_fwd = min(ts) * 1000
            fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))
            res_max = float(jnp.max(res))

            # Jacobian sur l'observable seulement
            fn_jac = jax.jit(jax.jacfwd(batched_obs))
            jac = fn_jac(params_true)
            jac.block_until_ready()

            ts = []
            for _ in range(3):
                t0 = time.time()
                jac = fn_jac(params_true)
                jac.block_until_ready()
                ts.append(time.time() - t0)

            t_jac = min(ts) * 1000
            jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
            has_nan = bool(jnp.any(jnp.isnan(jac)))

            ks_str = f"{ks}" if ks is not None else "—"
            nan_tag = " NaN!" if has_nan else ""
            wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""

            print(
                f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                f"{jac_err:10.2e}{nan_tag}  {res_max:10.2e}"
            )

Kerr oscillator: N=40, 1 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.



In [20]:
run_benchmark(
    "GMRES",
    gmres_single,
    [96],
)


GMRES
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err     res_max
  ----------------------------------------------------------------------------------


AttributeError: 'tuple' object has no attribute 'block_until_ready'